# Synthetic two-speaker conversations for caramel

Run from the repo root. A text teacher (Ollama or a local MLX model) writes the conversation, two Gradium Phonon voices
(on-device TTS) speak it, and the generator places both on one timeline with barge-ins, hold scenes (*wait*,
*listen*) and backchannels. caramel never sees text: the text and word timestamps only decide the timing.

**Output, chunk by chunk** (`CHUNK_SIZE` conversations at a time, so at most one chunk of audio is on disk):
1. Generate → `data/synthetic/episodes/NNNNNN/` (`assistant.wav` = speaker A, `user.wav` = speaker B, `annotations.json`).
2. Tokenize → Mimi tokens, one parquet shard per split, uploaded to `HF_REPO` as `data/{train,validation}-NNNNN.parquet`;
   `meta.json` in the repo records finished chunks and hours per split.
3. Clean up → the chunk's audio and the speech cache are deleted; only `annotations.json` stays as the resume marker.

`train.py --data hf://HF_REPO --dev hf://HF_REPO` streams the repo. Every conversation is trained from both sides.

### Resume an interrupted run
Keep `OUTPUT_DIR`, `HF_REPO` and the settings unchanged and rerun: uploaded chunks are skipped, finished episodes of the
current chunk are reused. Timed-out teacher requests go to `teacher_failures.jsonl`.

In [ ]:
!uv pip install openai datasets ptts mlx-vlm tqdm

In [ ]:
import sys
from pathlib import Path
from openai import OpenAI
from synthetic.generator import SyntheticConfig, SyntheticConversationGenerator
from synthetic.phonon_speech import PhononSpeech, PhononVoice, phonon_voices
from synthetic.teacher import ChatTeacher, MLXTeacher, load_prompt_corpus
from synthetic.hub import generate_to_hub

In [ ]:
OUTPUT_DIR = Path("data/synthetic")
HF_REPO = "your-username/caramel-synthetic"   # dataset repo (`hf auth login` first); created private if missing
CHUNK_SIZE = 100     # conversations per upload; at most one chunk of audio is ever on disk
MAX_SAMPLES = None # None = whole prompt corpus
SYSTEM_PROMPT = "You are JOSIE, a voice assistant. Your replies are spoken aloud: no markdown, code or math."

TEACHER = "ollama"   # "mlx": local MLX-VLM model, "ollama": OpenAI-compatible endpoint

if TEACHER == "mlx":
    teacher = MLXTeacher("mlx-community/gemma-4-e4b-it-4bit", max_tokens=512)
elif TEACHER == "ollama":
    # Fail visibly after a stalled request instead of waiting through SDK retries.
    client = OpenAI(
        base_url="http://localhost:11434/v1", api_key="ollama",
        timeout=120.0, max_retries=0,
    )
    teacher = ChatTeacher(client, assistant_model="mistral-large-4:cloud", followup_model="deepseek-v4.1-flash:cloud")
else:
    raise ValueError(f"Unknown TEACHER: {TEACHER}")

# phonon-40m-multilingual is gated (CC BY-NC 4.0): accept the terms on Hugging Face, then `hf auth login`.
# Voices: PhononVoice(name="Zoey") from `python -m ptts --list-voices`, PhononVoice(file="voice.safetensors"),
# or PhononVoice(wav="reference.wav") if the checkpoint supports cloning.
# User voice: one is picked at random per conversation from this pool. Select by name or filter, e.g.
#   phonon_voices("Zoey", "Hugh", "Maeve")  |  phonon_voices(accent="uk")  |  phonon_voices(gender="male")
ASSISTANT_VOICE = "Zoey"
USER_VOICES = phonon_voices(exclude=[ASSISTANT_VOICE])   # all 25 US + UK catalog voices except JOSIE's
speech = PhononSpeech(
    OUTPUT_DIR / "_speech_cache",
    model="hf://gradium/phonon-40m-multilingual",   # or "hf://gradium/phonon-31m-en" (Apache 2.0)
    language="en",
    user_voice=USER_VOICES,
    assistant_voice=PhononVoice(name=ASSISTANT_VOICE),
    min_alignment_match=0.8,
)
config = SyntheticConfig(
    max_turns=32, barge_in_probability=0.25, assistant_stop_ms=750, system_prompt=SYSTEM_PROMPT, seed=42,
    wait_probability=0.10,         # "can you wait a minute?" -> silent while the user is busy, then resume
    listen_probability=0.10,       # "listen until I finish my presentation" -> silent, then feedback
    max_wait_seconds=60,           # longest timed wait ("a minute")
    check_in_probability=0.5,      # "the minute is up, can I continue?" / "not yet, I'll tell you"
    backchannel_probability=0.2,   # "mhm", "right", "yeah" in the pauses of long user turns
)


In [ ]:
prompts, source = load_prompt_corpus("mlx-community/ultrafeedback-prompts-flat-rlhf")
generator = SyntheticConversationGenerator(teacher, speech, OUTPUT_DIR, config, source_metadata=source)
# Capture all thread stacks if TTS, alignment, or the teacher stops progressing.
# The log also works in kernels whose stderr does not expose a file descriptor.
import faulthandler
stall_log = OUTPUT_DIR / "generation_stall.log"
print(f"Stall diagnostics (after 120 seconds): {stall_log.resolve()}", flush=True)
with stall_log.open("w") as diagnostic_file:
    faulthandler.dump_traceback_later(120, repeat=True, file=diagnostic_file)
    try:
        meta = generate_to_hub(generator, prompts, HF_REPO, chunk_size=CHUNK_SIZE, max_samples=MAX_SAMPLES,
                               speech_cache=OUTPUT_DIR / "_speech_cache")
    finally:
        faulthandler.cancel_dump_traceback_later()
print(f"{len(meta['chunks'])} chunks, {meta['frames']['train'] / 12.5 / 3600:.2f} h train, {meta['frames']['validation'] / 12.5 / 3600:.2f} h validation on {HF_REPO}")

## Train
Streams the tokens straight from the Hub; nothing is downloaded to disk.

In [ ]:
print(f"python train.py --init ./caramel-1 --data hf://{HF_REPO} --dev hf://{HF_REPO} "
      "--fpb 1500 --accum 32 --epochs 3 --lr 3e-5 --warmup 10 --eval_every 25 --save_best --out runs/synthetic")